# Module 3.1 — Neural Differential Equations (Full Implementation)

This notebook provides a complete PyTorch implementation of Neural ODEs.
It accompanies the browser-based toy version in Module 3.1.

## Requirements
```
pip install torch torchdiffeq matplotlib numpy
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchdiffeq import odeint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Define the Neural ODE Model

A Neural ODE parameterizes the right-hand side of an ODE with a neural network:

$$\frac{dy}{dt} = f_\theta(t, y)$$

where $f_\theta$ is a neural network with parameters $\theta$.

In [ ]:
class ODEFunc(nn.Module):
    """Neural network defining the ODE right-hand side."""
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 2)
        )
        # Initialize weights small
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=0.1)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, t, y):
        return self.net(y)

## 2. Generate Training Data

We generate data from a known ODE system — the spiral ODE:

$$\frac{d}{dt}\begin{pmatrix} y_1 \\ y_2 \end{pmatrix} = \begin{pmatrix} -0.1 & 2.0 \\ -2.0 & -0.1 \end{pmatrix} \begin{pmatrix} y_1 \\ y_2 \end{pmatrix}$$

In [ ]:
# True dynamics: a decaying spiral
A_true = torch.tensor([[-0.1, 2.0], [-2.0, -0.1]])

def true_dynamics(t, y):
    return y @ A_true.T

# Generate ground truth trajectory
y0 = torch.tensor([[2.0, 0.0]])
t_span = torch.linspace(0, 5, 100)

with torch.no_grad():
    true_y = odeint(true_dynamics, y0, t_span, method='dopri5')

# Add noise for training data
noise_std = 0.05
train_y = true_y + noise_std * torch.randn_like(true_y)

# Visualize
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(true_y[:, 0, 0].numpy(), true_y[:, 0, 1].numpy(), 'b-', label='True')
plt.scatter(train_y[:, 0, 0].numpy(), train_y[:, 0, 1].numpy(), c='r', s=5, label='Noisy data')
plt.xlabel('$y_1$'); plt.ylabel('$y_2$')
plt.legend(); plt.title('Phase Portrait')

plt.subplot(1, 2, 2)
plt.plot(t_span.numpy(), true_y[:, 0, 0].numpy(), 'b-', label='$y_1$ true')
plt.plot(t_span.numpy(), true_y[:, 0, 1].numpy(), 'r-', label='$y_2$ true')
plt.xlabel('t'); plt.legend(); plt.title('Time Series')
plt.tight_layout()
plt.show()

## 3. Training Loop

Train the Neural ODE by:
1. Integrate the learned ODE forward from $y(0)$
2. Compute MSE loss against observed data
3. Backpropagate through the ODE solver using the adjoint method

In [ ]:
# Model and optimizer
func = ODEFunc(hidden_dim=64).to(device)
optimizer = torch.optim.Adam(func.parameters(), lr=1e-3)

# Training
n_epochs = 300
batch_time = 20  # Number of time steps per batch
losses = []

for epoch in range(n_epochs):
    optimizer.zero_grad()
    
    # Random starting index for batch
    s = np.random.choice(np.arange(len(t_span) - batch_time), size=1)[0]
    batch_y0 = train_y[s].to(device)
    batch_t = t_span[s:s+batch_time].to(device)
    batch_y = train_y[s:s+batch_time].to(device)
    
    # Forward pass: integrate ODE
    pred_y = odeint(func, batch_y0, batch_t, method='dopri5')
    
    # Loss
    loss = ((pred_y - batch_y) ** 2).mean()
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.6f}')

plt.figure(figsize=(8, 3))
plt.semilogy(losses)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Training Loss'); plt.grid(True, alpha=0.3)
plt.show()

## 4. Evaluate and Compare

In [ ]:
# Generate prediction over full time span
with torch.no_grad():
    pred_y = odeint(func, y0.to(device), t_span.to(device), method='dopri5').cpu()

# Extrapolation: predict beyond training data
t_extrap = torch.linspace(0, 10, 200)
with torch.no_grad():
    pred_extrap = odeint(func, y0.to(device), t_extrap.to(device), method='dopri5').cpu()
    true_extrap = odeint(true_dynamics, y0, t_extrap, method='dopri5')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(true_y[:, 0, 0], true_y[:, 0, 1], 'b-', lw=2, label='True')
axes[0].plot(pred_y[:, 0, 0], pred_y[:, 0, 1], 'r--', lw=2, label='Neural ODE')
axes[0].set_xlabel('$y_1$'); axes[0].set_ylabel('$y_2$')
axes[0].legend(); axes[0].set_title('Phase Portrait (Training)')

axes[1].plot(t_extrap, true_extrap[:, 0, 0], 'b-', label='True $y_1$')
axes[1].plot(t_extrap, pred_extrap[:, 0, 0], 'r--', label='Pred $y_1$')
axes[1].axvline(x=5, color='k', ls=':', label='Training boundary')
axes[1].legend(); axes[1].set_title('Extrapolation')

# Error analysis
error = torch.norm(pred_y - true_y, dim=-1).squeeze()
axes[2].plot(t_span, error)
axes[2].set_xlabel('t'); axes[2].set_ylabel('||error||')
axes[2].set_title('Prediction Error')

plt.tight_layout()
plt.show()

print(f'Mean L2 error (training range): {error.mean():.6f}')
extrap_error = torch.norm(pred_extrap - true_extrap, dim=-1).squeeze()
print(f'Mean L2 error (extrapolation): {extrap_error[100:].mean():.6f}')

## 5. Adjoint Method Visualization

The adjoint method computes gradients by solving a backward ODE:

$$\frac{da}{dt} = -a^T \frac{\partial f_\theta}{\partial y}, \quad \frac{d\theta}{dt} = -a^T \frac{\partial f_\theta}{\partial \theta}$$

This has O(1) memory cost regardless of the number of integration steps.

In [ ]:
# Compare memory usage: direct backprop vs adjoint
print('Neural ODE advantages over discrete ResNet:')
print(f'  - Parameters: {sum(p.numel() for p in func.parameters())} (shared across all depths)')
print(f'  - Memory: O(1) via adjoint method')
print(f'  - Adaptive computation: solver chooses evaluation points')
print(f'  - Continuous-time dynamics: interpolate at any t')